# 🤖 Financial Classifier: LLM Training & Comparison

**Objective**: Compare different Large Language Models (LLMs) to find the best classifier for financial transactions.

---

## 1. Setup and Environment

In [1]:
import os
import pandas as pd
import torch
try:
    # Fix for partially initialized 'torch' module in Windows environments
    if not hasattr(torch, 'version'):
        import torch.version 
except ImportError:
    pass
    
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ⚙️ Configuration
DATA_PATH = "../data/raw/db_mod_descript.csv"
OUTPUT_BASE_DIR = "../models/llm_comparison"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Using device: {device}")

c:\Users\diego\OneDrive - Universidad Rey Juan Carlos\Documentos\GIA_URJC\Curso 2025-26\Ap_IA\practicas\P2_AP-IA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Using device: cpu


## 2. Load and Prepare Data

In [2]:
# Load dataset
df = pd.read_csv(DATA_PATH).dropna(subset=['Description', 'Area'])

# Create label mapping
labels = sorted(df['Area'].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}
NUM_LABELS = len(labels)

print(f"📊 Categories found ({NUM_LABELS}): {labels}")

# Map labels to IDs
df['label'] = df['Area'].map(label2id)

# Split data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print(f"✅ Data split: {len(train_df)} training, {len(test_df)} testing samples.")

📊 Categories found (9): ['Deposit', 'Food', 'Food, Vacations', 'Investment', 'Invoice', 'Invoice, Vacations', 'Leisure', 'Leisure, Vacations', 'Salary']
✅ Data split: 708 training, 177 testing samples.


## 3. Training Loop Definition

In [3]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro")
    }

def train_model(model_name, model_checkpoint):
    print(f"\n🔄 --- Training Model: {model_name} ({model_checkpoint}) ---")
    
    # Load Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=False)
    
    def tokenize_function(examples):
        # Use truncation and padding to handle descriptions correctly
        return tokenizer(examples["Description"], truncation=True, max_length=64, padding=False)

    # Prepare Datasets
    train_dataset = Dataset.from_pandas(train_df[['Description', 'label']].reset_index(drop=True))
    test_dataset = Dataset.from_pandas(test_df[['Description', 'label']].reset_index(drop=True))
    
    # CRITICAL: We MUST remove 'Description' column after tokenization 
    # because the Trainer cannot process non-tensor string columns.
    tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["Description"])
    tokenized_test = test_dataset.map(tokenize_function, batched=True, remove_columns=["Description"])
    
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # Load Model
    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint, 
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id
    ).to(device)

    # Training Arguments
    training_args = TrainingArguments(
        output_dir=os.path.join(OUTPUT_BASE_DIR, model_name),
        learning_rate=2e-5,
        per_device_train_batch_size=8, # Reduced for stability on local machines
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        report_to="none"
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    results = trainer.evaluate()
    return results

## 4. Run Comparison

**Note**: This process can take several minutes per model on CPU.

In [4]:
MODELS_TO_TEST = {
    "DistilBERT-Multi": "distilbert-base-multilingual-cased",
    "BETO (Spanish)": "dccuchile/bert-base-spanish-wwm-cased",
    "XLM-RoBERTa": "xlm-roberta-base"
}

all_results = []

for name, checkpoint in MODELS_TO_TEST.items():
    try:
        res = train_model(name, checkpoint)
        all_results.append({
            "Model": name,
            "Accuracy": res["eval_accuracy"],
            "F1-Macro": res["eval_f1_macro"]
        })
    except Exception as e:
        print(f"❌ Failed to train {name}: {e}")


🔄 --- Training Model: DistilBERT-Multi (distilbert-base-multilingual-cased) ---


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4995.95it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.577679,0.903955,0.609095
2,No log,0.124902,1.000000,1.000000
3,No log,0.077911,1.000000,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.01it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


❌ Failed to train DistilBERT-Multi: on_train_begin must be called before on_evaluate

🔄 --- Training Model: BETO (Spanish) (dccuchile/bert-base-spanish-wwm-cased) ---


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 9649.51it/s]
BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTE

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## 5. View Results

In [ ]:
if all_results:
    results_df = pd.DataFrame(all_results).sort_values(by="F1-Macro", ascending=False)
    print("\n🏆 Final Comparison Table:")
    display(results_df)

    # Plot
    plt.figure(figsize=(10, 6))
    sns.barplot(data=results_df.melt(id_vars='Model'), x='Model', y='value', hue='variable')
    plt.title("LLM Performance Comparison")
    plt.ylabel("Score")
    plt.ylim(0, 1.1)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()
else:
    print("⚠️ No results to show. Check errors above.")

## 6. Inference Winner

In [ ]:
def quick_predict(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_class_id = logits.argmax().item()
    return model.config.id2label[predicted_class_id]

print("💡 The models are saved in '../models/llm_comparison/'. You can load the winner for production!")